In [1]:
import pandas as pd
import numpy as np
import os
import shutil
import re
from pathlib import Path

In [36]:
def pivot_table(df, index_col, col, val):
    pivot_table_df = pd.pivot_table(df, index=index_col, columns=col, values=val, aggfunc="mean", fill_value=0).stack().reset_index()
    if isinstance(index_col, list):
        new_cols = index_col + [col, "Average Elapsed Minutes"]
    else:
        new_cols = [index_col, col, "Average Elapsed Minutes"]
    pivot_table_df.columns = new_cols
    pivot_table_df["Average Elapsed Minutes"] = pivot_table_df["Average Elapsed Minutes"].round(2)
    df["Failed Jobs"] = (df["Status Code"] > 1).astype(int)
    df["Successful Jobs"] = (df["Status Code"] <= 1).astype(int)
    job_status_summary = df.groupby(index_col + [col])[["Failed Jobs", "Successful Jobs"]].sum().reset_index()
    pivot_table_df = pivot_table_df.merge(job_status_summary, on=index_col + [col], how="left")
    return pivot_table_df

In [37]:
policies_list_df = pd.read_csv("data\email_dl.csv")

In [38]:
source_folders = [Path(r"C:\Users\User\Desktop\Projects\netbackup_weekly_report\data\May_2026")]

all_dfs = []

for source_folder in source_folders:
    report_csv_files = source_folder.rglob("daily_backup_report_*")
    for report_csv_file in report_csv_files:
        df = pd.read_csv(report_csv_file)
        df = df.drop(columns = 'Number of Child Jobs')
        if 'Failure Count' in df.columns and 'Backup Selection' in df.columns:
            df = df.drop(columns=['Failure Count', 'Backup Selection'], errors='ignore')
        all_dfs.append(df)

# combine all csv into one dataframe
may_df = pd.concat(all_dfs, ignore_index=True)
may_df["Start Time"] = pd.to_datetime(may_df["Start Time"], errors="coerce")
may_df["End Time"] = pd.to_datetime(may_df["End Time"], errors="coerce")
may_df["Elapsed Minutes"] = pd.to_timedelta(may_df['Elapsed Time'])
may_df["Elapsed Minutes"] = (may_df['Elapsed Minutes'].dt.total_seconds() / 60).round(2)

In [39]:
may_df

,Job,Type,Schedule,Parent Job,Client,Policy,State,Status Code,Start Time,End Time,Elapsed Time,Elapsed Minutes
0,8804053,DBBACKUP,-,-,anmirny11.dsta.infra.sg,CATALOG-BACKUP-DTTA,DONE,1,2026-04-30 09:00:00,2026-04-30 09:35:57,35m57s,35.95
1,8804054,BACKUP,-,-,anmirny11.dsta.infra.sg,EHAB-DB-SQL-MYClearance-INTEL,DONE,0,2026-04-30 09:00:00,2026-04-30 09:01:47,1m47s,1.78
2,8804055,BACKUP,Daily-Log-9am,8804054,ocurewera1.dsta.gov.sg,EHAB-DB-SQL-MYClearance-INTEL,DONE,0,2026-04-30 09:00:06,2026-04-30 09:01:47,1m41s,1.68
3,8804056,DBBACKUP,Daily-Catalog,8804053,anmirny11.dsta.infra.sg,CATALOG-BACKUP-DTTA,DONE,0,2026-04-30 09:00:11,2026-04-30 09:06:48,6m37s,6.62
4,8804057,BACKUP,Daily-Log-9am,8804055,OCUREWERA1,EHAB-DB-SQL-MYClearance-INTEL,DONE,0,2026-04-30 09:00:22,2026-04-30 09:00:39,17s,0.28
...,...,...,...,...,...,...,...,...,...,...,...,...
103232,8911417,BACKUP,Daily-Log-8am,8911331,OCDMITROV01,EHAB-DB-SQL-WEBAPP1,DONE,0,2026-05-31 08:16:19,2026-05-31 08:16:34,15s,0.25
103233,8911418,BACKUP,Daily-Log-8am,8911331,OCDMITROV01,EHAB-DB-SQL-WEBAPP1,DONE,0,2026-05-31 08:16:41,2026-05-31 08:16:56,15s,0.25
103234,8911419,BACKUP,Daily-Log-8am,8911331,OCDMITROV01,EHAB-DB-SQL-WEBAPP1,DONE,0,2026-05-31 08:17:04,2026-05-31 08:17:22,18s,0.30
103235,8911420,BACKUP,Daily-Log-8am,8911331,OCDMITROV01,EHAB-DB-SQL-WEBAPP1,DONE,0,2026-05-31 08:17:31,2026-05-31 08:17:48,17s,0.28


In [42]:
policy_df = {}
policy_summary_df = {}

output_path = Path("report_output/")
if not output_path.exists():
    output_path.mkdir()

for pattern in policies_list_df['Policy'].dropna().unique():
    filtered_df = may_df[
        may_df['Policy'].astype(str).str.contains(
            pattern,
            case=False,
            na=False,
            regex=True
        )].copy()
    filtered_df = filtered_df.loc[filtered_df['Parent Job'] == '-']
    filtered_df = filtered_df.sort_values(by=['Client', 'Policy'], ascending=True)
    filtered_pivot_df = pivot_table(filtered_df.copy(), index_col=["Client", "Policy"], col="State", val="Elapsed Minutes")
    policy_df[pattern] = filtered_df
    policy_summary_df[pattern] = filtered_pivot_df

In [43]:
# for name, df in policy_df.items():
#     df.to_csv(f"{output_path}/{name}_report.csv", index=False)

for name in policy_df.keys():
    with pd.ExcelWriter(f"{output_path}/{name}_report.xlsx", engine="openpyxl") as writer:

        policy_df[name].to_excel(
            writer,
            sheet_name=f"{name}_Report",
            index=False
        )

        policy_summary_df[name].to_excel(
            writer,
            sheet_name=f"{name}_Summary",
            index=False
        )